In [3]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

In [7]:
# load data

BASE_DIR = Path().resolve().parent

PROCESSED_PATH = BASE_DIR / "data" / "processed" / "creditcard_clean.csv"
MODEL_PATH = BASE_DIR / "artifacts" / "fraud_model.joblib"

df = pd.read_csv(PROCESSED_PATH)
model = joblib.load(MODEL_PATH)

In [8]:
# split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [9]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [10]:
metrics_summary = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "f2": fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob),
    "log_loss": log_loss(y_test, y_prob),
    "mcc": matthews_corrcoef(y_test, y_pred),
    "cohen_kappa": cohen_kappa_score(y_test, y_pred)
}

In [11]:
metrics_summary

{'accuracy': 0.9994889507630493,
 'balanced_accuracy': 0.8894030764542643,
 'precision': 0.9024390243902439,
 'recall': 0.7789473684210526,
 'f1': 0.8361581920903954,
 'f2': 0.8008658008658008,
 'roc_auc': 0.9726970955127843,
 'pr_auc': 0.8224615631900266,
 'log_loss': 0.003089111587969073,
 'mcc': 0.8381744656331769,
 'cohen_kappa': 0.8359036510284429}

In [12]:
pd.DataFrame(metrics_summary.items(), columns=["metric", "value"])

,metric,value
0,accuracy,0.999489
1,balanced_accuracy,0.889403
2,precision,0.902439
3,recall,0.778947
4,f1,0.836158
5,f2,0.800866
6,roc_auc,0.972697
7,pr_auc,0.822462
8,log_loss,0.003089
9,mcc,0.838174


In [13]:
import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score

thresholds = np.arange(0.10, 0.95, 0.05)

rows = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)

    rows.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_test, y_pred_t, zero_division=0),
        "recall": recall_score(y_test, y_pred_t, zero_division=0),
        "f1": f1_score(y_test, y_pred_t, zero_division=0),
        "f2": fbeta_score(y_test, y_pred_t, beta=2, zero_division=0)
    })

threshold_df = pd.DataFrame(rows)
threshold_df

,threshold,precision,recall,f1,f2
0,0.10,0.706422,0.810526,0.754902,0.787321
1,0.15,0.783505,0.800000,0.791667,0.796646
2,0.20,0.835165,0.800000,0.817204,0.806794
3,0.25,0.835165,0.800000,0.817204,0.806794
4,0.30,0.853933,0.800000,0.826087,0.810235
5,0.35,0.860465,0.778947,0.817680,0.793991
6,0.40,0.891566,0.778947,0.831461,0.799136
7,0.45,0.902439,0.778947,0.836158,0.800866
8,0.50,0.902439,0.778947,0.836158,0.800866
9,0.55,0.901235,0.768421,0.829545,0.791757


Best Threshold is 0.3

In [14]:
threshold = 0.30
y_pred_custom = (y_prob >= threshold).astype(int)

In [15]:
print(classification_report(y_test, y_pred_custom, zero_division=0))
print(confusion_matrix(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.85      0.80      0.83        95

    accuracy                           1.00     56746
   macro avg       0.93      0.90      0.91     56746
weighted avg       1.00      1.00      1.00     56746

[[56638    13]
 [   19    76]]


Threshold tuning was applied to optimize the tradeoff between precision and recall. While the default threshold of 0.5 yielded high precision, lowering the threshold to 0.30 improved recall, allowing the model to detect more fraudulent transactions while maintaining acceptable precision. This demonstrates the importance of threshold selection in imbalanced classification problems such as fraud detection.

In [ ]:
import joblib
import json
from datetime import datetime
from pathlib import Path


# Define project paths
BASE_DIR = Path().resolve().parent
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
version = "1.0"

In [18]:
# Create metadata
metadata = {
    "version": version,
    "model_type": "XGBoost",
    "library": "xgboost",
    "training_date": datetime.now().strftime("%Y-%m-%d"),

    "dataset": {
        "name": "creditcard.csv",
        "n_samples": int(len(X_train)),
        "fraud_cases": int((y_train == 1).sum()),
        "non_fraud_cases": int((y_train == 0).sum())
    },

    "performance": {
        "accuracy": 0.9995,
        "balanced_accuracy": 0.8894,
        "precision": 0.9024,
        "recall": 0.7789,
        "f1_score": 0.8362,
        "f2_score": 0.8009,
        "roc_auc": 0.9727,
        "pr_auc": 0.8225,
        "log_loss": 0.0031,
        "mcc": 0.8382,
        "cohen_kappa": 0.8359
    },

    "threshold": {
        "value": 0.30,
        "reason": "Optimized to improve recall while maintaining strong precision"
    },

    "features": {
        "input_features": [
            *[f"V{i}" for i in range(1, 29)],
            "Amount_log",
            "Time_sin",
            "Time_cos"
        ],
        "target": "Class"
    },

    "description": (
        "Fraud detection model using PCA-transformed features (V1-V28), "
        "log-transformed transaction amount, and cyclical time encoding. "
        "XGBoost selected based on highest PR AUC and balanced precision-recall tradeoff."
    )
}

In [19]:
# 5. Save metadata
metadata_path = ARTIFACTS_DIR / f"fraud_model_v{version}_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# 6. Confirm paths
print("Metadata saved to:", metadata_path)

Metadata saved to: C:\Users\tevin\OneDrive\Desktop\fraud_detection\artifacts\fraud_model_v1.0_metadata.json
